# Food Delivery Operations & Restaurant Service Performance Analytics
### Complete EDA — Analysis of `train.csv` (45,584 rows, 20 columns)
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Plotly (optional).  
**Outputs:** `food_delivery_cleaned.csv`, `eda_summary.csv`, and 15 charts inside `eda_charts/`.

**How to run in VS Code:** Put this notebook and `train.csv` in the project root folder, select your `.venv` Python kernel, then click **Run All**. Run cells in order. Charts are displayed and also saved as PNG files.

**Scope:** Dataset has *delivery operations* but **no restaurant names, revenue, or restaurant-level ratings**; avoid claims about individual restaurant profitability.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 110})
DATA_FILE = Path("train.csv")
CHART_DIR = Path("eda_charts")
CHART_DIR.mkdir(exist_ok=True)
assert DATA_FILE.exists(), f"Place train.csv in the notebook folder: {Path.cwd()}"
print("Working folder:", Path.cwd())
print("Dataset:", DATA_FILE.resolve())

## 1. Load and inspect the raw data

In [ ]:
raw = pd.read_csv(DATA_FILE)
print("Shape:", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.head())
print("Duplicates:", raw.duplicated().sum())
display(pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum(), "missing_pct": raw.isna().mean().mul(100).round(2)}).sort_values("missing", ascending=False))

## 2. Clean and standardize (actual uploaded CSV column names)
Preserve original nulls where numeric values are missing; label unknown categorical records as `Unknown`. Exclude impossible latitudes/longitudes when computing distance.

In [ ]:
df = raw.copy()
df.columns = df.columns.str.strip()
rename = {
    "ID": "order_id", "Delivery_person_ID": "delivery_person_id",
    "Delivery_person_Age": "delivery_person_age", "Delivery_person_Ratings": "delivery_person_rating",
    "Restaurant_latitude": "restaurant_latitude", "Restaurant_longitude": "restaurant_longitude",
    "Delivery_location_latitude": "delivery_latitude", "Delivery_location_longitude": "delivery_longitude",
    "Order_Date": "order_date", "Time_Orderd": "time_ordered", "Time_Order_picked": "time_picked",
    "Weather_conditions": "weather", "Road_traffic_density": "road_traffic_density",
    "Vehicle_condition": "vehicle_condition", "Type_of_order": "type_of_order",
    "Type_of_vehicle": "type_of_vehicle", "multiple_deliveries": "multiple_deliveries",
    "Festival": "festival", "City": "city_type", "Time_taken (min)": "delivery_minutes"
}
df = df.rename(columns=rename)
required = ["delivery_minutes", "weather", "road_traffic_density", "type_of_order", "type_of_vehicle"]
assert all(c in df for c in required), f"Missing columns: {[c for c in required if c not in df]}"
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace({"": pd.NA, "NaN": pd.NA, "nan": pd.NA, "NULL": pd.NA, "null": pd.NA, "None": pd.NA})
for col in ["delivery_person_age", "delivery_person_rating", "multiple_deliveries", "delivery_minutes"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
for col in ["weather", "road_traffic_density", "type_of_order", "type_of_vehicle", "festival", "city_type"]:
    df[col] = df[col].fillna("Unknown")
df["order_date"] = pd.to_datetime(df["order_date"], format="%d-%m-%Y", errors="coerce")
# Times in this dataset are strings like HH:MM, sometimes missing.
def to_minutes(s):
    time = pd.to_datetime(s, format="%H:%M", errors="coerce")
    return time.dt.hour * 60 + time.dt.minute

df["order_clock_minutes"] = to_minutes(df["time_ordered"])
df["pickup_clock_minutes"] = to_minutes(df["time_picked"])
df["order_hour"] = (df["order_clock_minutes"] // 60).astype("Int64")
df["order_weekday"] = df["order_date"].dt.day_name()
df["order_month"] = df["order_date"].dt.to_period("M").astype("string")
# A pickup may be after midnight; this calculates elapsed clock minutes, not total delivery duration.
df["order_to_pickup_minutes"] = (df["pickup_clock_minutes"] - df["order_clock_minutes"]) % 1440
df.loc[df["order_clock_minutes"].isna() | df["pickup_clock_minutes"].isna(), "order_to_pickup_minutes"] = np.nan
# Remove duplicate order IDs, then invalid delivery times.
before = len(df)
df = df.drop_duplicates(subset=["order_id"]).copy()
df = df.loc[df["delivery_minutes"].between(1, 240)].copy()
print("Rows before:", before, "Rows after:", len(df))
print("Column check:", df[required].head().to_string(index=False))

## 3. Geographical feature engineering
The Haversine distance is an approximate **straight-line** distance, not actual road distance. Coordinates of zero or outside valid geographic bounds are treated as missing.

In [ ]:
coord_cols = ["restaurant_latitude", "restaurant_longitude", "delivery_latitude", "delivery_longitude"]
for c in coord_cols: df[c] = pd.to_numeric(df[c], errors="coerce")
valid = (df["restaurant_latitude"].between(-90,90) & df["delivery_latitude"].between(-90,90)
        & df["restaurant_longitude"].between(-180,180) & df["delivery_longitude"].between(-180,180)
        & df[coord_cols].ne(0).all(axis=1))
lat1 = np.radians(df["restaurant_latitude"])
lat2 = np.radians(df["delivery_latitude"])
dlat = lat2 - lat1
dlon = np.radians(df["delivery_longitude"] - df["restaurant_longitude"])
a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
km = 6371 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
df["straight_line_km"] = km.where(valid & (km <= 100))
df["delivery_band"] = pd.cut(df["delivery_minutes"], bins=[0,20,35,60,240], labels=["<=20 min", "21–35 min", "36–60 min", ">60 min"])
print("Valid calculated distances:", df["straight_line_km"].notna().sum())
print("Order date range:", df["order_date"].min(), "to", df["order_date"].max())

## 4. Key performance indicators and data quality

In [ ]:
summary = pd.DataFrame({"Metric": ["Delivery records", "Mean delivery (min)", "Median delivery (min)", "90th percentile (min)", "Slow deliveries (>35 min)", "Unique delivery people", "Distance records available"],
"Value": [len(df), round(df.delivery_minutes.mean(),2), round(df.delivery_minutes.median(),2), round(df.delivery_minutes.quantile(.9),2), round(df.delivery_minutes.gt(35).mean()*100,2), df.delivery_person_id.nunique(), int(df.straight_line_km.notna().sum())]})
display(summary)
summary.to_csv("eda_summary.csv", index=False)
display(df.select_dtypes(include="number").describe().T.round(2))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_after_cleaning"))

def save_chart(name):
    plt.tight_layout()
    plt.savefig(CHART_DIR / f"{name}.png", bbox_inches="tight", dpi=150)
    plt.show()
    plt.close()


## 5. Fifteen EDA visualizations
Each plot is calculated from your cleaned CSV, not sample figures. Comparisons are descriptive; they do **not** by themselves prove causation.

In [ ]:
# Chart 1 — Delivery time histogram
plt.hist(df["delivery_minutes"].dropna(), bins=35, edgecolor="white")
plt.title("1. Delivery Time Distribution")
plt.xlabel("Delivery minutes")
save_chart("01_delivery_time_distribution")

In [ ]:
# Chart 2 — Delivery performance bands
order = ["<=20 min", "21–35 min", "36–60 min", ">60 min"]
sns.countplot(color="steelblue", data=df, x="delivery_band", order=order)
plt.title("2. Delivery Duration Categories")
plt.xlabel("Duration band")
save_chart("02_delivery_bands")

In [ ]:
# Chart 3 — Orders by type
sns.countplot(color="steelblue", data=df, y="type_of_order", order=df.type_of_order.value_counts().index)
plt.title("3. Orders by Food Category")
save_chart("03_order_types")

In [ ]:
# Chart 4 — Traffic impact
traffic_order = [x for x in ["Low", "Medium", "High", "Jam", "Unknown"] if x in df.road_traffic_density.unique()]
sns.barplot(color="steelblue", data=df, x="road_traffic_density", y="delivery_minutes", order=traffic_order, estimator="mean", errorbar=None)
plt.title("4. Average Delivery Time by Traffic Density")
plt.ylabel("Mean delivery minutes")
save_chart("04_traffic_delivery")

In [ ]:
# Chart 5 — Weather boxplots
plt.figure(figsize=(11,5))
sns.boxplot(color="steelblue", data=df, x="weather", y="delivery_minutes", showfliers=False, order=df.weather.value_counts().index)
plt.xticks(rotation=25)
plt.title("5. Delivery Time by Weather")
save_chart("05_weather_boxplot")

In [ ]:
# Chart 6 — Vehicle performance
sns.barplot(color="steelblue", data=df, y="type_of_vehicle", x="delivery_minutes", estimator="mean", errorbar=None, order=df.type_of_vehicle.value_counts().index)
plt.title("6. Average Delivery Time by Vehicle Type")
plt.xlabel("Mean delivery minutes")
save_chart("06_vehicle_performance")

In [ ]:
# Chart 7 — City type comparison
sns.countplot(color="steelblue", data=df, x="city_type", order=df.city_type.value_counts().index)
plt.title("7. Delivery Records by City Type")
plt.xticks(rotation=20)
save_chart("07_city_types")

In [ ]:
# Chart 8 — Daily order counts
series = df.groupby("order_date").size().sort_index()
plt.plot(series.index, series.values)
plt.title("8. Daily Delivery Counts")
plt.xlabel("Order date")
plt.ylabel("Delivery count")
plt.xticks(rotation=25)
save_chart("08_daily_volume")

In [ ]:
# Chart 9 — Orders by hour
hours = df.order_hour.dropna().astype(int).value_counts().reindex(range(24),fill_value=0)
plt.bar(hours.index, hours.values)
plt.xticks(range(0,24,2))
plt.xlabel("Hour of day")
plt.ylabel("Delivery count")
plt.title("9. Orders by Time of Day")
save_chart("09_hourly_orders")

In [ ]:
# Chart 10 — Delivery time by food category
sns.barplot(color="steelblue", data=df, y="type_of_order", x="delivery_minutes", errorbar=None, estimator="mean")
plt.title("10. Mean Delivery Time by Food Category")
plt.xlabel("Mean delivery minutes")
save_chart("10_food_delivery_time")

In [ ]:
# Chart 11 — Weather × traffic
pivot = df.pivot_table(index="weather", columns="road_traffic_density", values="delivery_minutes", aggfunc="mean", observed=True)
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("11. Mean Delivery Minutes: Weather × Traffic")
save_chart("11_weather_traffic_heatmap")

In [ ]:
# Chart 12 — Courier ratings versus delivery time
sample = df[["delivery_person_rating", "delivery_minutes"]].dropna().sample(n=min(4000,df[["delivery_person_rating", "delivery_minutes"]].dropna().shape[0]),random_state=42)
sns.scatterplot(color="steelblue", data=sample, x="delivery_person_rating", y="delivery_minutes", alpha=.25, s=12)
plt.title("12. Courier Rating vs Delivery Time (sample)")
save_chart("12_rating_vs_time")

In [ ]:
# Chart 13 — Approximate distance versus delivery time
dist_sample = df[["straight_line_km", "delivery_minutes"]].dropna()
dist_sample = dist_sample.sample(n=min(5000,len(dist_sample)), random_state=42)
sns.scatterplot(color="steelblue", data=dist_sample, x="straight_line_km", y="delivery_minutes", s=12, alpha=.25)
plt.title("13. Straight-line Distance vs Delivery Time (sample)")
plt.xlabel("Approximate straight-line distance (km)")
save_chart("13_distance_time")

In [ ]:
# Chart 14 — Weekday delivery time
weekdays = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
sns.barplot(color="steelblue", data=df, x="order_weekday", y="delivery_minutes", order=weekdays, estimator="mean", errorbar=None)
plt.xticks(rotation=35)
plt.ylabel("Mean delivery minutes")
plt.title("14. Average Delivery Time by Weekday")
save_chart("14_weekday_comparison")

In [ ]:
# Chart 15 — Numeric correlations
cols = ["delivery_minutes", "delivery_person_age", "delivery_person_rating", "multiple_deliveries", "vehicle_condition", "straight_line_km", "order_to_pickup_minutes"]
plt.figure(figsize=(10,7))
sns.heatmap(df[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("15. Correlation Matrix (Pearson)")
save_chart("15_correlation_heatmap")

## 6. Export cleaned data and summary
The dashboard can read `food_delivery_cleaned.csv` from this project folder. Missing numeric data remains missing rather than being replaced by fabricated measurements.

In [ ]:
output = Path("food_delivery_cleaned.csv")
df.to_csv(output, index=False)
print("Saved:", output.resolve())
print("Chart images:", len(list(CHART_DIR.glob('*.png'))), "in", CHART_DIR.resolve())
print("Summary:", Path("eda_summary.csv").resolve())
print("Data shape:", df.shape)
display(df.head())

## 7. Interpretation prompts for project report
1. Which traffic category has the highest **average** delivery duration? Verify category sample sizes before making claims.
2. Which weather category shows the widest variation in delivery duration?
3. When are orders most frequent? Note that records represent this historical dataset, not all current delivery demand.
4. Does straight-line distance correlate with delivery duration? It is not a driving-route distance.
5. Are observed differences large enough to be meaningful? Correlation and group averages do not establish causation.

**Caveat:** This project analyzes food-delivery *operations*, not actual restaurant sales or restaurant-by-restaurant profit, because the CSV lacks those fields.
